In [6]:
%pip install --upgrade --quiet  langchain    --quiet
%pip install --upgrade --quiet  bitsandbytes --quiet
%pip install weaviate-client --quiet
%pip install sentence-transformers --quiet
%pip install -qU langchain langchain-openai langchain-anthropic langchain-community
%pip install python-dotenv

In [7]:
import weaviate

# import weaviate.classes as wvc
from dotenv import load_dotenv, find_dotenv
import os
from langchain.vectorstores.weaviate import Weaviate
from langchain.embeddings.huggingface import HuggingFaceEmbeddings


load_dotenv(find_dotenv("tokens.env"))
WCS_API_KEY = os.getenv("YOUR_WEAVIATE_KEY")
WCS_CLUSTER_URL = os.getenv("YOUR_WEAVIATE_CLUSTER")

client = weaviate.Client(
    url=WCS_CLUSTER_URL,
    auth_client_secret=weaviate.auth.AuthApiKey(WCS_API_KEY),
)
device = "cuda"
embed_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"device": device, "batch_size": 32},
)

vectorstore = Weaviate(
    client,
    index_name="LangChain_0c70358e34034236ba8f84cd318e2c7b",
    embedding=embed_model,
    text_key="text",
    by_text=False,
    attributes=["title", "authors", "pmid_id", "journal"],
)
query = "children with benign childhood epilepsy"
docs = vectorstore.similarity_search_with_score(query)
print(docs)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/93.0k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[(Document(page_content='Childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes are the most common forms of benign epilepsy syndromes. Although cognitive dysfunctions occur in children with both childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes, the similarity between their patterns of underlying cognitive impairments is not well understood. To describe these patterns, we examined multiple cognitive functions in children with childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes.', metadata={'_additional': {'vector': [-0.003304629, 0.0036504886, 0.0050439346, 0.07229095, -0.04269789, 0.06180551, 0.056088503, 0.030591605, 0.015711457, -0.026703918, 0.0246742, -0.024723371, 0.057477407, 0.017496426, -0.016508639, -0.037786607, -0.024544494, 0.002425198, -0.004426411, -0.0044447193, 0.00047287132, -0.0073750787, 0.0425874, -0.007882363, 0.023320608, 0.0068156277, 0.020889113, 0.008847715, -0.0196

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You're a helpful AI assistant. Given a user question and some pubmed article snippets, answer the user question using only the information contained in the article snippets. Important: Name the title of the article snippet you used to generate the answer. If none of the articles answer the question, just say you don't know.\n\nHere are the pubmed articles:{context}",
        ),
        ("human", "{question}"),
    ]
)
prompt.pretty_print()

================================ System Message ================================

You're a helpful AI assistant. Given a user question and some pubmed article snippets, answer the user question using only the information contained in the article snippets. Important: Name the title of the article snippet you used to generate the answer. If none of the articles answer the question, just say you don't know.

Here are the pubmed articles:{context}

================================ Human Message =================================

{question}


In [9]:
from operator import itemgetter
from typing import List

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)


def format_docs(docs: List[Document]) -> str:
    """Convert Documents to a single string.:"""
    formatted = [
        f"Article Title: {doc.metadata['title']}\nArticle Snippet: {doc.page_content}"
        for doc in docs
    ]
    return "\n\n" + "\n\n".join(formatted)


format = itemgetter("docs") | RunnableLambda(format_docs)
# subchain for generating an answer once we've done retrieval
answer = prompt | llm | StrOutputParser()
# complete chain that calls wiki -> formats docs to string -> runs answer subchain -> returns just the answer and retrieved docs.
chain = (
    RunnableParallel(question=RunnablePassthrough(), docs=vectorstore.as_retriever())
    .assign(context=format)
    .assign(answer=answer)
    .pick(["answer", "docs"])
)

In [10]:
chain.invoke(
    "what are the most common forms of benign epilepsy syndromes for children?"
)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}